In [21]:
import os
import yaml
import base64
import fitz  # PyMuPDF (페이지를 이미지로 변환하기 위해 필요)
import warnings; warnings.filterwarnings('ignore')
from dotenv import load_dotenv

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore

In [22]:
load_dotenv(dotenv_path="../.env") 
with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

gemini_api_key = os.getenv(config['api_keys']['google_gemini_env_name'])
if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키 로드 완료")
else:
    print("❌ API 키 오류")

✅ Gemini API 키 로드 완료


In [18]:
# ==========================================
# 🔍 [Path A] 텍스트 검색 (Parent Document Retriever 적용)
# ==========================================
pdf_path = "../data/raw/samsung_report.pdf"

print("⏳ 고급 텍스트 검색기(Parent Document Retriever)를 구축합니다...")
loader = PyMuPDFLoader(pdf_path)
# 문서 로드 (PyMuPDF는 1페이지 = 1개의 Document 객체로 로드됩니다. 이것이 'Parent'가 됩니다.)
documents = loader.load()

# 1. 자식 분할기 (Child Splitter): 아주 정밀한 검색을 위해 400자 단위로 잘게 쪼갭니다.
child_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)

# 2. 비어있는 Vector DB 생성 (여기에 잘게 쪼개진 자식 조각들이 들어갑니다)
embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
vectorstore = Chroma(
    collection_name="split_parents", 
    embedding_function=embedding_model
)

# 3. 원본 저장소 (Store): 잘리기 전의 원본 페이지(Parent)들을 통째로 보관할 메모리 창고입니다.
store = InMemoryStore()

# 4. 부모 문서 검색기 조립!
retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

# 문서를 추가하면, 내부적으로 알아서 자식으로 쪼개서 Vector DB에 넣고 원본은 Store에 저장합니다.
print("📚 문서를 정밀 분할하고 부모-자식 관계를 매핑하는 중... (10~20초 소요)")
retriever.add_documents(documents)

# 검색 시 상위 3개의 '부모 문서(전체 페이지)'를 가져오도록 설정
retriever.search_kwargs = {"k": 3} 

print("✅ 고급 하이브리드 Vector DB 구축 완료!")

⏳ 고급 텍스트 검색기(Parent Document Retriever)를 구축합니다...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28424.82it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📚 문서를 정밀 분할하고 부모-자식 관계를 매핑하는 중... (10~20초 소요)
✅ 고급 하이브리드 Vector DB 구축 완료!


In [32]:
# ==========================================
# 🎯 검색 및 데이터 준비
# ==========================================
query = "2024년 삼성전자의 당기순이익은 얼마인가요?"
print(f"\n🧐 질문: {query}")

# 1. 텍스트 검색 실행 -> "페이지 전체 텍스트(Parent)"가 통째로 반환됩니다!
retrieved_parents = retriever.invoke(query)

target_pages = set()
parent_texts = []

# 검색된 페이지와 다음 페이지(N+1)를 타겟으로 삼고, 전체 텍스트를 수집합니다.
for doc in retrieved_parents:
    p_num = doc.metadata.get('page', 0)
    target_pages.add(p_num)
    target_pages.add(p_num + 1) # 표가 잘렸을 경우를 대비한 N+1 안전장치
    
    # 쪼개지지 않은 온전한 페이지 전체 텍스트를 저장!
    parent_texts.append(f"[페이지 {p_num + 1} 원본 전체 텍스트]\n{doc.page_content}")

target_pages = sorted(list(target_pages))
text_context = "\n\n".join(parent_texts)

print(f"🎯 텍스트 검색 기반 캡처 대상 페이지: {target_pages} (1부터 시작하는 기준)")

# 2. 타겟 페이지를 모두 고화질 이미지로 캡처
print("📸 타겟 페이지들을 고화질 이미지로 캡처합니다...")
doc_pdf = fitz.open(pdf_path)
encoded_images = []

for p_num in target_pages:
    if p_num < len(doc_pdf):
        page = doc_pdf.load_page(p_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(2.0, 2.0))
        encoded_images.append(base64.b64encode(pix.tobytes("png")).decode('utf-8'))

print("✅ 다중 페이지 이미지 변환 및 텍스트 문맥 준비 완료!")


🧐 질문: 2024년 삼성전자의 당기순이익은 얼마인가요?
🎯 텍스트 검색 기반 캡처 대상 페이지: [264, 265, 292, 293, 467, 468] (1부터 시작하는 기준)
📸 타겟 페이지들을 고화질 이미지로 캡처합니다...
✅ 다중 페이지 이미지 변환 및 텍스트 문맥 준비 완료!


In [30]:
# ==========================================
# 🧠 [최종 융합] VLM에 '전체 텍스트'와 '전체 이미지' 동시 주입
# ==========================================
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 프롬프트: 텍스트는 힌트로 쓰고, 확정은 이미지로 하라고 강력하게 통제합니다.
prompt = f"""당신은 꼼꼼한 금융 데이터 분석가입니다.
아래 제공된 [전체 페이지 텍스트]와 첨부된 [고화질 문서 이미지들]을 모두 교차 검증하여 질문에 답하세요.

[전체 페이지 텍스트 (Parent Document)]
{text_context}

[질문]
{query}

[행동 지침 - 매우 중요!]
1. 텍스트는 표의 대략적인 위치나 맥락을 파악하는 '참고용 힌트'로만 사용하세요. 표 구조가 깨져있을 수 있습니다.
2. 최종적인 수치(숫자)와 단위는 반드시 첨부된 [문서 이미지]의 표를 직접 눈으로 확인하고 검증한 뒤 답변하세요.
3. 표가 여러 페이지에 나뉘어 있을 수 있으므로 모든 이미지를 종합적으로 분석하세요.
"""

# 메시지 리스트에 텍스트 먼저 담고, 이미지들을 차례대로 이어 붙입니다.
content_list = [{"type": "text", "text": prompt}]
for b64 in encoded_images:
    content_list.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}})

message = HumanMessage(content=content_list)

print("🧠 Gemini 2.5 Flash가 전체 텍스트 문맥과 이미지를 동시에 분석 중입니다...\n")
response = llm.invoke([message])

print("=" * 50)
print("🤖 최종 AI 답변 (Parent Doc + Multi-Image Dual-Path):")
print(response.content)
print("=" * 50)

🧠 Gemini 2.5 Flash가 전체 텍스트 문맥과 이미지를 동시에 분석 중입니다...

🤖 최종 AI 답변 (Parent Doc + Multi-Image Dual-Path):
제공된 [전체 페이지 텍스트]와 [고화질 문서 이미지들]을 모두 교차 검증한 결과, 2024년 삼성전자의 당기순이익은 명시적으로 기재되어 있지 않습니다.

다만, 2024년 연결기준 매출 301조원, 영업이익 33조원, 그리고 ROE(자기자본이익률) 9.0%는 명시되어 있습니다. 재무상태표에서는 2024년(제56기) 연결기준 자본총계가 402,192,070 백만원으로 확인됩니다.

당기순이익은 ROE와 자본총계를 통해 추정할 수 있으나, 질문은 명시된 수치를 요구하므로 직접적인 답변은 어렵습니다.


In [25]:
print("\n" + "="*50)
print("🧐 AI에게 어려운 질문을 던져봅시다 (하이브리드 다중 페이지 테스트)")

query = "2024년 삼성전자의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

# --- 1단계: 텍스트 검색으로 연관 페이지 번호 싹쓸이 ---
print("🔍 [Path A 작동] 텍스트 키워드로 관련 페이지들을 찾는 중...")
retrieved_docs = retriever.invoke(query)

# 검색된 상위 3개 문서에서 고유한 페이지 번호를 추출하고, 
# 표가 다음 페이지로 넘어갔을 확률을 고려해 바로 다음 페이지(+1)도 후보에 넣습니다.
target_pages = set()
for doc in retrieved_docs:
    p_num = doc.metadata.get('page', 0)
    target_pages.add(p_num)
    target_pages.add(p_num + 1)
    target_pages.add(p_num + 2)
    target_pages.add(p_num + 3)

# 오름차순으로 정렬 (예: [41, 42, 43])
target_pages = sorted(list(target_pages))
print(f"🎯 텍스트 검색 기반 타겟 페이지들: {target_pages} (0부터 시작)")

# --- 2단계: 알아낸 여러 페이지를 모두 고화질 '이미지'로 변환하기 ---
print(f"📸 [Path B 작동] 총 {len(target_pages)}장의 페이지를 이미지로 캡처합니다...")

doc = fitz.open(pdf_path)
encoded_images = [] # 여러 장의 이미지를 담을 리스트

for p_num in target_pages:
    if p_num < len(doc): # 문서의 전체 페이지 수를 넘지 않도록 안전장치
        page = doc.load_page(p_num)
        zoom_matrix = fitz.Matrix(2.0, 2.0)
        pix = page.get_pixmap(matrix=zoom_matrix)
        
        img_bytes = pix.tobytes("png")
        b64 = base64.b64encode(img_bytes).decode('utf-8')
        encoded_images.append(b64)

print("✅ 다중 페이지 이미지 캡처 및 변환 완료!")


🧐 AI에게 어려운 질문을 던져봅시다 (하이브리드 다중 페이지 테스트)
질문: 2024년 삼성전자의 당기순이익은 얼마인가요?

🔍 [Path A 작동] 텍스트 키워드로 관련 페이지들을 찾는 중...
🎯 텍스트 검색 기반 타겟 페이지들: [264, 265, 266, 267, 292, 293, 294, 295, 467, 468, 469, 470] (0부터 시작)
📸 [Path B 작동] 총 12장의 페이지를 이미지로 캡처합니다...
✅ 다중 페이지 이미지 캡처 및 변환 완료!


In [26]:
# ==========================================
# 🧠 [최종 융합] VLM에게 여러 장의 이미지와 텍스트를 동시 제공
# ==========================================
print("🧠 Gemini 2.5 Flash에게 검색된 '여러 장의 이미지'를 모두 제공하여 스스로 표를 찾게 합니다...\n")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 추출했던 텍스트 내용들을 모두 합치기
text_context = "\n\n".join([doc.page_content for doc in retrieved_docs])

# 프롬프트 메시지 조립 시작 (텍스트 먼저 넣기)
content_list = [
    {
        "type": "text", 
        "text": f"""당신은 꼼꼼한 금융 데이터 분석가입니다.
아래 제공된 [텍스트 컨텍스트]와 뒤이어 첨부된 [여러 장의 문서 이미지]를 모두 훑어보고 사용자의 [질문]에 답변해주세요.
설명 텍스트와 표가 서로 다른 페이지에 나뉘어 있을 수 있습니다. 모든 이미지를 교차 검증하여 정확한 수치를 찾으세요.

[텍스트 컨텍스트 (참고용)]
{text_context}

[질문]
{query}
"""
    }
]

# 캡처한 여러 장의 이미지를 메시지에 차례대로 이어 붙이기!
for b64 in encoded_images:
    content_list.append({
        "type": "image_url", 
        "image_url": {"url": f"data:image/jpeg;base64,{b64}"}
    })

# 최종 메시지 객체 생성
message = HumanMessage(content=content_list)

# AI 호출
response = llm.invoke([message])

print("=" * 50)
print("🤖 최종 AI 답변 (Multi-Image Dual-Path RAG):")
print(response.content)
print("=" * 50)

🧠 Gemini 2.5 Flash에게 검색된 '여러 장의 이미지'를 모두 제공하여 스스로 표를 찾게 합니다...

🤖 최종 AI 답변 (Multi-Image Dual-Path RAG):
2024년 삼성전자의 당기순이익은 **34조 4,514억원**입니다.

이는 첨부된 문서 이미지 중 'Page 293'의 '나. 영업실적' 표에서 '제56기'의 '당기순이익' 항목과 해당 페이지 하단의 설명 텍스트에서 확인할 수 있습니다.
*   표: 34,451,351 백만원
*   설명 텍스트: "당기순이익은 전년 대비 18조 9,643억원(122.5%) 증가한 **34조 4,514억원**을 기록하였습니다."
